In [ ]:
"""
feature_engineering.py
======================
Phase 1 – Part B: Convert raw monthly transaction data
into behavioral user-level profiles for ML training.

Input:  data/raw/finance_data.csv   (NO segment column — leakage-free)
Output: data/processed/features.csv (2000 rows, one per user)

Every feature here represents a BEHAVIOR signal, not a raw number.
ML will learn patterns from these signals, not from rupee amounts.
"""

import numpy as np
import pandas as pd
from scipy.stats import linregress
import os

# ─────────────────────────────────────────────
# LOAD DATA
# ─────────────────────────────────────────────

def load_data(path="data/raw/finance_data.csv"):
    df = pd.read_csv(path)
    print(f"✅ Loaded: {len(df):,} rows, {df['user_id'].nunique():,} users")
    return df


# ─────────────────────────────────────────────
# FEATURE COMPUTATION (grouped by user)
# ─────────────────────────────────────────────

def compute_features(df):
    """
    For each user, aggregate 24 monthly records into one
    behavioral profile row.

    Features are grouped into 4 categories:
      A) Income stability
      B) Expense behavior
      C) Savings behavior
      D) Financial stress indicators
    """

    feature_rows = []

    for user_id, group in df.groupby("user_id"):
        g = group.sort_values("month").reset_index(drop=True)

        inc  = g["income"].values
        exp  = g["total_expense"].values
        sav  = g["savings"].values
        irr  = g["irregular_expense"].values

        n = len(g)   # should be 24

        # Safe income denominator — prevents division by zero if a clamped
        # month somehow produces a near-zero income value
        inc_safe = inc + 1e-6

        # ── A: INCOME STABILITY ────────────────────────────────────────────

        avg_income        = np.mean(inc)
        income_volatility = np.std(inc) / (avg_income + 1e-6)   # coefficient of variation (normalized)

        # Linear trend of income over months (positive = growing income)
        months = np.arange(1, n + 1)
        income_slope, _, _, _, _ = linregress(months, inc)
        income_growth_rate = income_slope / (avg_income + 1e-6)   # normalized slope

        # ── B: EXPENSE BEHAVIOR ────────────────────────────────────────────

        expense_ratio_mean  = np.mean(exp / inc_safe)            # avg fraction of income spent
        expense_ratio_std   = np.std(exp / inc_safe)             # how much that fraction varies
        expense_volatility  = np.std(exp) / (np.mean(exp) + 1e-6)   # normalized expense variation

        # Category concentrations (what % of expenses goes where)
        avg_rent_pct       = np.mean(g["rent"].values / inc_safe)
        avg_food_pct       = np.mean(g["food"].values / inc_safe)
        avg_travel_pct     = np.mean(g["travel"].values / inc_safe)

        # Irregular expense behavior
        irregular_freq     = np.mean(irr > 0)               # fraction of months with shock expense
        avg_irregular_amt  = np.mean(irr[irr > 0]) if np.any(irr > 0) else 0.0

        # ── C: SAVINGS BEHAVIOR ────────────────────────────────────────────

        savings_ratio_mean  = np.mean(sav / inc_safe)            # avg savings rate
        savings_ratio_std   = np.std(sav / inc_safe)             # consistency of savings
        savings_volatility  = np.std(sav) / (abs(np.mean(sav)) + 1e-6)  # normalized, avoid div/0

        # Running cumulative savings — proxy for financial buffer
        cumulative_savings  = np.cumsum(sav)
        financial_buffer    = cumulative_savings[-1] / (avg_income + 1e-6)   # buffer in months of income
        # Note: buffer_volatility removed — std of a cumulative series is a math
        # artifact (measures curve spread, not behavior). savings_volatility
        # already captures month-to-month savings swings correctly.

        # ── D: FINANCIAL STRESS INDICATORS ────────────────────────────────

        # % of months where user spent more than they earned
        neg_savings_freq    = np.mean(sav < 0)

        # % of months with severe overspending (expense > 120% of income)
        severe_overspend_freq = np.mean((exp / inc_safe) > 1.2)

        # Longest consecutive streak of negative savings
        neg_streak = 0
        max_neg_streak = 0
        for s in sav:
            if s < 0:
                neg_streak += 1
                max_neg_streak = max(max_neg_streak, neg_streak)
            else:
                neg_streak = 0

        # Low savings months: < 5% savings rate (not negative, but dangerously low)
        low_savings_freq    = np.mean((sav / inc_safe) < 0.05)

        # Static info
        age       = g["age"].iloc[0]
        city_tier = g["city_tier"].iloc[0]

        feature_rows.append({
            "user_id": user_id,
            "age": age,
            "city_tier": city_tier,

            # Income
            "avg_income":           round(avg_income, 2),
            "income_volatility":    round(income_volatility, 4),
            "income_growth_rate":   round(income_growth_rate, 6),

            # Expense
            "expense_ratio_mean":   round(expense_ratio_mean, 4),
            "expense_ratio_std":    round(expense_ratio_std, 4),
            "expense_volatility":   round(expense_volatility, 4),
            "avg_rent_pct":         round(avg_rent_pct, 4),
            "avg_food_pct":         round(avg_food_pct, 4),
            "avg_travel_pct":       round(avg_travel_pct, 4),
            "irregular_freq":       round(irregular_freq, 4),
            "avg_irregular_amt":    round(avg_irregular_amt, 2),

            # Savings
            "savings_ratio_mean":   round(savings_ratio_mean, 4),
            "savings_ratio_std":    round(savings_ratio_std, 4),
            "savings_volatility":   round(savings_volatility, 4),
            "financial_buffer":     round(financial_buffer, 4),

            # Stress
            "neg_savings_freq":         round(neg_savings_freq, 4),
            "severe_overspend_freq":    round(severe_overspend_freq, 4),
            "max_neg_savings_streak":   int(max_neg_streak),
            "low_savings_freq":         round(low_savings_freq, 4),
        })

    return pd.DataFrame(feature_rows)


# ─────────────────────────────────────────────
# ENCODE CITY TIER
# ─────────────────────────────────────────────

def encode_city_tier(df):
    """
    Convert city_tier string to ordinal number.
    metro=3, tier2=2, tier3=1
    Higher city tier = higher cost of living context.
    """
    mapping = {"metro": 3, "tier2": 2, "tier3": 1}
    df["city_tier_code"] = df["city_tier"].map(mapping)
    return df


# ─────────────────────────────────────────────
# VALIDATION
# ─────────────────────────────────────────────

def validate_features(df):
    print("\n── FEATURE ENGINEERING VALIDATION ─────────────")
    print(f"  Total users      : {len(df):,}")
    print(f"  Feature count    : {len(df.columns) - 3}")   # exclude user_id, city_tier, city_tier_code
    print(f"  Any nulls        : {df.isnull().sum().sum()}")
    print(f"  Any inf values   : {np.isinf(df.select_dtypes('number').values).sum()}")
    print()

    numeric = df.select_dtypes(include='number').drop(columns=["user_id", "city_tier_code"])
    print("  Feature ranges:")
    for col in numeric.columns:
        print(f"    {col:<30} min={numeric[col].min():>10.4f}  max={numeric[col].max():>10.4f}  mean={numeric[col].mean():>8.4f}")

    # ── CORRELATION CHECK ──────────────────────────────────────────────────
    # Pairs with |correlation| > 0.85 are likely redundant.
    # Flag them now so Phase 2 ML can decide which to drop.
    print()
    print("  Highly correlated feature pairs (|r| > 0.85) — review before ML:")
    corr = numeric.corr().abs()
    found = False
    for i in range(len(corr.columns)):
        for j in range(i + 1, len(corr.columns)):
            r = corr.iloc[i, j]
            if r > 0.85:
                print(f"    ⚠  {corr.columns[i]:<30} ↔  {corr.columns[j]:<30}  r={r:.3f}")
                found = True
    if not found:
        print("    ✅ No highly correlated pairs found.")
    print("────────────────────────────────────────────────\n")


# ─────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────

if __name__ == "__main__":
    print("\n🔄 Running Feature Engineering (Phase 1B)...\n")

    os.makedirs("data/processed", exist_ok=True)

    # Load clean data (no segment column)
    df_raw = load_data("data/raw/finance_data.csv")

    # Compute behavioral features
    print("  Computing behavioral features per user...")
    df_features = compute_features(df_raw)

    # Encode city tier
    df_features = encode_city_tier(df_features)

    # ── FEATURE SELECTION ─────────────────────────────────────────────────
    # Drop mathematically redundant columns identified by correlation analysis.
    # Reason for each drop is documented below.
    #
    #   savings_ratio_mean  → r=1.000 with expense_ratio_mean (literally 1 - expense)
    #   savings_ratio_std   → r=1.000 with expense_ratio_std (same reason)
    #   expense_ratio_std   → captures variance already visible in expense_volatility
    #   financial_buffer    → r=1.000 with expense_ratio_mean (scaled savings transform)
    #   avg_rent_pct        → r=0.945 with expense_ratio_mean (component of it)
    #   avg_food_pct        → r=0.958 with expense_ratio_mean (component of it)
    #   avg_travel_pct      → r=0.867 with expense_ratio_mean (component of it)
    #   low_savings_freq    → r=0.980 with neg_savings_freq (near-identical signal)
    #   severe_overspend_freq → r=0.895 with neg_savings_freq (same stress signal)
    #
    # All computation is still above — only the output is trimmed.
    # If you need a dropped feature later, uncomment it here.

    FEATURES_TO_DROP = [
        "savings_ratio_mean",
        "savings_ratio_std",
        "expense_ratio_std",
        "financial_buffer",
        "avg_rent_pct",
        "avg_food_pct",
        "avg_travel_pct",
        "low_savings_freq",
        "severe_overspend_freq",
    ]

    df_features = df_features.drop(columns=FEATURES_TO_DROP)

    print(f"  Features after redundancy removal: {len(df_features.columns) - 3} "
          f"(dropped {len(FEATURES_TO_DROP)} redundant columns)\n")

    # Validate
    validate_features(df_features)

    # Save
    output_path = "data/processed/features.csv"
    df_features.to_csv(output_path, index=False)

    print(f"✅ Features saved → {output_path}")
    print(f"   Shape: {df_features.shape[0]:,} rows × {df_features.shape[1]} columns")
    print()
    print("📌 Next: Use features.csv for risk labeling + ML training (Phase 2)")
    print("⚠️  Do NOT add 'segment' back to this file.\n")